In [12]:
import os
import torch
import pandas as pd
from epiweeks import Week
import preprocess_data as prep
import model as mc
import matplotlib.pyplot as plt 

pd.options.mode.chained_assignment = None

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

regioes_estados = {
        'Sul': ['SC', 'PR', 'RS'],
        'Sudeste': ['SP', 'MG', 'RJ', 'ES'],
        'Nordeste': ['BA', 'CE', 'PE', 'PB', 'PI', 'RN', 'MA', 'AL', 'SE'],
        'Centro-Oeste': ['DF', 'MT', 'MS', 'GO'],
        'Norte': ['RO', 'AC', 'AM', 'RR', 'PA', 'AP', 'TO']
    } 
    
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

columns_to_normalize = ['casos','epiweek', 'biome']

predict_n = 36
max_epiweek = 16
    
boxcox = False

TEST_YEAR = 2023

In [13]:
batch_size = 1
doenca = 'chikungunya'
min_delta = 0.0
patience= 20
lr = 5e-4
min_year = 2015
model_name = 'dengue_base'

filename = f'./data/{doenca}.csv.gz'

df = prep.load_cases_data(filename)

df = df.loc[df.epiweek <= int(f'{TEST_YEAR}{max_epiweek}')]

enso = prep.load_enso_weekly()

enso_neutro = prep.load_enso_weekly(filename='data/enso_weekly_neutro.csv')

enso = None 
enso_neutro = None

Run for state: 

for region in regioes_estados.keys(): 

    print(region)

    label = f'{region}_{TEST_YEAR-1}_{model_name}'

    for state in regioes_estados[region]: 

        df_reg = df.loc[df.uf ==state]
        df_reg = df_reg.loc[df_reg.index >= pd.to_datetime(Week(2015,1).startdate())]
        

        model = mc.load_model(
                        region,
                        TEST_YEAR,
                        doenca,
                        model_name,
                        predict_n,
                        max_epiweek,
                        device
                    )  

        filename_el_nino = f'predictions/preds_{doenca}_{state}_{TEST_YEAR}_{model_name}.csv'

        if os.path.exists(filename_el_nino):
            df_preds = pd.read_csv(filename_el_nino)
            df_preds.date = pd.to_datetime(df_preds.date)
        else:
            df_preds =  mc.sum_regions_predictions(model, df_reg, enso, TEST_YEAR, columns_to_normalize,
                                            max_epiweek = max_epiweek,
                                            boxcox = boxcox,
                                            n_passes = 500,
                                            min_year= min_year, media = True)

                
            df_preds.to_csv(filename_el_nino)

        if enso_neutro is not None: 
            filename_neutro = f'predictions/preds_{doenca}_{state}_{TEST_YEAR}_neutro_{model_name}.csv'

            if os.path.exists(filename_neutro):
                df_preds_neutro = pd.read_csv(filename_neutro)
                df_preds_neutro.date = pd.to_datetime(df_preds_neutro.date)
            else:

                df_preds_neutro =  mc.sum_regions_predictions(model, df_reg, enso_neutro, TEST_YEAR, columns_to_normalize,
                                                max_epiweek = max_epiweek,
                                                boxcox = boxcox,
                                                n_passes = 500,
                                                min_year= min_year, media = True)

                    
                df_preds_neutro.to_csv(filename_neutro)


Run for region: 

In [14]:
for region in regioes_estados.keys(): 

    print(region)

    label = f'{region}_{TEST_YEAR-1}_{model_name}'

    df_reg = df.loc[df.uf.isin(regioes_estados[region])]
    df_reg = df_reg.loc[df_reg.index >= pd.to_datetime(Week(2015,1).startdate())]
    
    if 'dengue' in model_name: 

        model_name_n = model_name[7:]

        model = mc.load_model(
                    region,
                    TEST_YEAR,
                    'dengue',
                    model_name_n,
                    predict_n,
                    max_epiweek,
                    device
                )

    else: 
        model = mc.load_model(
                    region,
                    TEST_YEAR,
                    doenca,
                    model_name,
                    predict_n,
                    max_epiweek,
                    device
                )  
            
    

    filename_el_nino = f'predictions/preds_{doenca}_{region}_{TEST_YEAR}_{model_name}.csv'

    if os.path.exists(filename_el_nino):
        df_preds = pd.read_csv(filename_el_nino)
        df_preds.date = pd.to_datetime(df_preds.date)
    else:
        df_preds =  mc.sum_regions_predictions(model, df_reg, enso, TEST_YEAR, columns_to_normalize,
                                        max_epiweek = max_epiweek,
                                        boxcox = boxcox,
                                        n_passes = 500,
                                        min_year= min_year, media = True)

            
        df_preds.to_csv(filename_el_nino)

    if enso_neutro is not None: 
        filename_neutro = f'predictions/preds_{doenca}_{region}_{TEST_YEAR}_neutro_{model_name}.csv'

        if os.path.exists(filename_neutro):
            df_preds_neutro = pd.read_csv(filename_neutro)
            df_preds_neutro.date = pd.to_datetime(df_preds_neutro.date)
        else:

            df_preds_neutro =  mc.sum_regions_predictions(model, df_reg, enso_neutro, TEST_YEAR, columns_to_normalize,
                                            max_epiweek = max_epiweek,
                                            boxcox = boxcox,
                                            n_passes = 500,
                                            min_year= min_year, media = True)

                
            df_preds_neutro.to_csv(filename_neutro)


Sul
Sudeste
Nordeste
Centro-Oeste
Norte


### Total cases

df_end = pd.DataFrame()
for region in regioes_estados.keys(): 

    print(region)

    label = f'{region}_{TEST_YEAR-1}_{model_name}'

    df_reg = df.loc[df.uf.isin(regioes_estados[region])]
    df_reg = df_reg.loc[df_reg.index >= pd.to_datetime(Week(2015,1).startdate())]


    if 'dengue' in model_name: 

        model_name_n = model_name[7:]

        model = mc.load_model(
                    region,
                    TEST_YEAR,
                    'dengue',
                    model_name_n,
                    predict_n,
                    max_epiweek,
                )

    else: 
        model = mc.load_model(
                    region,
                    TEST_YEAR,
                    doenca,
                    model_name,
                    predict_n,
                    max_epiweek,
                )  
            
    
    preds =  mc.sum_regions_predictions(model, df_reg, enso, TEST_YEAR, columns_to_normalize,
                                    max_epiweek = max_epiweek,
                                    boxcox = boxcox,
                                    n_passes = 500,
                                    min_year= min_year, media = True, return_samples= True)
    

    df_sum_el_nino = mc.get_total_cases(preds, region, model_name= 'El niño')

    if enso_neutro is not None: 

        preds_neutro =  mc.sum_regions_predictions(model, df_reg, enso_neutro, TEST_YEAR, columns_to_normalize,
                                        max_epiweek = max_epiweek,
                                        boxcox = boxcox,
                                        n_passes = 500,
                                        min_year= min_year, media = True, return_samples=True)
        
        df_sum_neutro = mc.get_total_cases(preds_neutro, region, model_name= 'Neutro')

        
        df_end = pd.concat([df_end, df_sum_el_nino, df_sum_neutro], ignore_index= True)
    else: 
        df_end = pd.concat([df_end, df_sum_el_nino], ignore_index= True)


df_end.to_csv(f'predictions/total_cases_all_regions_{doenca}_{model_name}.csv', index = False)